# 04 LEAR

This notebook analyses the LEAR benchmark for hourly Dutch day-ahead prices.

What LEAR does:
- LEAR stands for a linear autoregressive model with exogenous regressors
- in this thesis workstream it is implemented as a LASSO-based linear model so coefficients can shrink toward zero
- this makes LEAR a strong structured benchmark between naive rules and more flexible machine-learning models

Feature sets used now:
- `FS1`: lagged DA prices only
- `FS2`: lagged DA prices plus forecast-known calendar features

Leakage prevention:
- the model is refit separately for each forecast origin
- only history strictly before the `08:00` D-1 origin is used
- recursive prediction is used to bridge the hours between the origin and later horizon timestamps
- the target column stays observed-only for scoring, while lag construction uses the deterministic feature-source series to avoid source-gap propagation

Later extension point:
- external feature families are not in `FS2`
- they will enter through the later `FS3` family-testing workflow

In [1]:
from pathlib import Path
import pandas as pd

run_root = Path('data/02_Forecasting/01_DA_prices/hourly_da/runs')
latest_run = sorted(run_root.glob('*_lear_benchmark'))[-1]
latest_run

IndexError: list index out of range

In [ ]:
metrics_overall = pd.read_csv(latest_run / 'metrics_overall.csv')
metrics_by_lead_day = pd.read_csv(latest_run / 'metrics_by_lead_day.csv')
timing_summary = pd.read_csv(latest_run / 'origin_timing_summary.csv')
dm_results = pd.read_csv(latest_run / 'diebold_mariano_results.csv')

display(metrics_overall)
display(metrics_by_lead_day[metrics_by_lead_day['model'].str.startswith('lear_')])
display(timing_summary[timing_summary['model'].str.startswith('lear_')])

Interpretation guide:

- Lower `MAE`, `RMSE`, and `rmae_vs_official_naive` are better.
- `rmae_vs_official_naive < 1` means the model beats the official naive benchmark.
- `bias` shows systematic over- or under-forecasting.
- `fit_time_warning_count` should remain low; repeated warnings would signal an operational burden for daily refitting.
- Diebold-Mariano results help test whether average error differences against the official naive are statistically meaningful.

In [ ]:
dm_results[dm_results['challenger_model'].str.startswith('lear_')]